[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrite/kernels/blob/main/labs/stage-3/lab-3.4-masks-as-loop-structure.ipynb)

# LAB·3.4 · Skipping blocks: masks as loop structure

**Hardware:** correctness anywhere; the sparsity-scaling measurement on TPU.

The second axis of hard kernels: data-dependent iteration. A causal mask is not something you add to scores; it is blocks you never visit. Here the kv loop bound *depends on the query block index*, so masked blocks cost nothing. Splash generalizes this from causal structure to arbitrary block masks via scalar prefetch; same idea, more machinery.

In [ ]:
import time
import numpy as np
import jax
import jax.numpy as jnp
from jax.experimental import pallas as pl

print(jax.__version__, jax.devices())
ON_TPU = jax.devices()[0].platform == "tpu"
INTERP = not ON_TPU  # interpret mode anywhere; compiled kernels on a real TPU

def check(name, got, want, tol=2e-2):
    err = float(jnp.abs(got.astype(jnp.float32) - want.astype(jnp.float32)).max())
    status = "ok" if err <= tol else "FAIL"
    print(f"{name}: max err {err:.3e} [{status}]")
    assert err <= tol, name


In [ ]:
import functools

def causal_flash_kernel(q_ref, k_ref, v_ref, o_ref, *, block_q, block_kv):
    qi = pl.program_id(0)
    q = q_ref[...].astype(jnp.float32)
    row0 = qi * block_q

    def step(j, state):
        m, l, acc = state
        kb = k_ref[pl.ds(j * block_kv, block_kv), :].astype(jnp.float32)
        vb = v_ref[pl.ds(j * block_kv, block_kv), :].astype(jnp.float32)
        s = q @ kb.T
        cols = j * block_kv + jax.lax.broadcasted_iota(jnp.int32, s.shape, 1)
        rows = row0 + jax.lax.broadcasted_iota(jnp.int32, s.shape, 0)
        s = jnp.where(cols <= rows, s, -jnp.inf)      # edge block: mask inside
        m_new = jnp.maximum(m, jnp.max(s, axis=-1))
        alpha = jnp.exp(m - m_new)
        p = jnp.exp(s - m_new[:, None])
        l_new = l * alpha + jnp.sum(p, axis=-1)
        acc_new = acc * alpha[:, None] + p @ vb
        return m_new, l_new, acc_new

    # the causal structure IS the loop bound: blocks past the diagonal never run
    n_live = (row0 + block_q + block_kv - 1) // block_kv
    m0 = jnp.full((block_q,), -jnp.inf, jnp.float32)
    l0 = jnp.zeros((block_q,), jnp.float32)
    acc0 = jnp.zeros((block_q, v_ref.shape[1]), jnp.float32)
    m, l, acc = jax.lax.fori_loop(0, n_live, step, (m0, l0, acc0))
    o_ref[...] = (acc / l[:, None]).astype(o_ref.dtype)

In [ ]:
def causal_flash(q, k, v, block_q=64, block_kv=64):
    sq, d = q.shape
    return pl.pallas_call(
        functools.partial(causal_flash_kernel, block_q=block_q, block_kv=block_kv),
        grid=(sq // block_q,),
        in_specs=[
            pl.BlockSpec((block_q, d), lambda i: (i, 0)),
            pl.BlockSpec(k.shape, lambda i: (0, 0)),
            pl.BlockSpec(v.shape, lambda i: (0, 0)),
        ],
        out_specs=pl.BlockSpec((block_q, d), lambda i: (i, 0)),
        out_shape=jax.ShapeDtypeStruct(q.shape, q.dtype),
        interpret=INTERP,
    )(q, k, v)

n = 512
q = jax.random.normal(jax.random.key(0), (n, 64))
k = jax.random.normal(jax.random.key(1), (n, 64))
v = jax.random.normal(jax.random.key(2), (n, 64))
mask = jnp.tril(jnp.ones((n, n), bool))
ref = jax.nn.softmax(jnp.where(mask, q @ k.T, -jnp.inf), axis=-1) @ v
check("causal flash", causal_flash(q, k, v), ref, tol=1e-4)

## Measure the scaling (TPU runtime)

Causal skips about half the blocks, so against your dense LAB·3.2 kernel expect close to 2x at long sequence. Then the real exercise: replace the causal bound with an arbitrary *block mask* array (say, sliding window plus a few global columns) passed in as data, and make the loop follow it. When that works, you have rebuilt the idea inside Splash, and the speedup scales with whatever sparsity your mask has.